In [90]:
import os
import re
import pickle ,  uuid
from typing import List, Dict, Any, Optional
from pathlib import Path
import numpy as np
from tqdm import tqdm

try:
    import fitz 
except ImportError:
    fitz = None

try:
    import pdfplumber
except ImportError:
    pdfplumber = None

from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase

In [91]:
os.chdir("/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject")
os.getcwd()

'/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject'

In [92]:
from src.Config.config import NEO4J_PASSWORD , NEO4J_URI , NEO4J_USERNAME , EMBEDDING_MODEL

In [93]:
NEO4J_URI = NEO4J_URI
NEO4J_USERNAME = NEO4J_USERNAME
NEO4J_PASSWORD = NEO4J_PASSWORD

EMBEDDING_MODEL_NAME = EMBEDDING_MODEL
EMBEDDING_DEVICE = "cuda"

DATA_FOLDER = "Datasets/Unstructured_Data"

CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300

CRIME_KEYWORDS = ["يعاقب", "جريمة", "جنحة", "جناية"]
# small curated lists — expand for better coverage
LEGAL_TERMS = [
    "قتل", "سرقة", "حيازة", "اتجار", "رشوة", "اعتداء", "تزوير",
    "تهريب", "مخدر", "سلاح", "تحريض", "اغتصاب", "اختطاف", "غسل أموال"
]
PENALTY_KEYWORDS = [
    "سجن مؤبد", "سجن", "حبس", "غرامة", "الإعدام", "الحدّ", "الحبس", "سنة", "سنوات", "شهر", "أشهر" 
]
EXCEPTION_KEYWORDS = ["ما لم", "باستثناء", "لا يسري", "لا يسرى", "ما عدا", "لا تطبق","لا تسري", "لا" , "مع عدم الإخلال" , "يعاقب" ]

ARTICLE_PATTERNS = [r'((?:المادة|مادة)\s*\d+)', r'((?:المواد|المادة)\s*\d+)']
REFERENCE_PATTERN = r"المادة\s*\(?\d+\)?"

SENTENCE_SPLIT = r'(?<=[\.\n؛؟!])\s*'

In [94]:
# ------------------------------
def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    # remove invisible / bidi markers
    text = re.sub(r'[\u200c\u200b\u202c\u200e\u200f\ufeff]', '', text)
    # normalize newlines
    text = re.sub(r'\r\n|\r', '\n', text)
    # remove tatweel
    text = text.replace('ـ', '')
    # collapse multiple spaces/newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # trim
    return text.strip()


def load_pdf_pymupdf(path: str) -> Optional[str]:
    """Attempt to load PDF using PyMuPDF (fitz). Returns joined pages or None."""
    if 'fitz' not in globals() or fitz is None:
        return None
    try:
        doc = fitz.open(path)
        pages = []
        for page in doc:
            txt = page.get_text()
            if txt and txt.strip():
                pages.append(txt)
        doc.close()
        return "\n\n".join(pages) if pages else None
    except Exception:
        return None


def load_pdf_pdfplumber(path: str) -> Optional[str]:
    """Attempt to load PDF using pdfplumber as a fallback."""
    if 'pdfplumber' not in globals() or pdfplumber is None:
        return None
    try:
        pages = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                txt = page.extract_text()
                if txt and txt.strip():
                    pages.append(txt)
        return "\n\n".join(pages) if pages else None
    except Exception:
        return None


def load_document(path: str) -> Optional[Dict[str, Any]]:
    """Load .pdf or .txt and return a normalized dict {filename,path,text,law_name} or None."""
    ext = Path(path).suffix.lower()
    filename = Path(path).name
    raw_text = None

    if ext == ".pdf":
        raw_text = load_pdf_pymupdf(path) or load_pdf_pdfplumber(path)
    elif ext == ".txt":
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                raw_text = f.read()
        except Exception:
            raw_text = None
    else:
        return None

    if not raw_text:
        return None

    normalized = normalize_arabic(raw_text)
    # default law_name from file stem (can be overridden later)
    return {"filename": filename, "path": str(path), "text": normalized, "law_name": Path(path).stem}


# ------------------------------
def extract_entities_from_text(article: Dict[str, Any]) -> Dict[str, Any]:
    """
    Heuristic entity extractor for a single article or text-block.
    Expects article to have: 'text', 'article_id' and 'law_name' (optional).
    Returns dict with lists:
      { 'concepts': [str,...],
        'penalties': [ {penalty_type, raw, article_id}, ... ],
        'exceptions': [ {exception_text, article_id}, ... ],
        'references': [ {from_article_id, to_article_id, reference_text}, ... ] }
    """
    text = article.get("text", "") or ""
    article_id = article.get("article_id")
    law_name = article.get("law_name") or ""

    out = {"concepts": [], "penalties": [], "exceptions": [], "references": []}

    # Penalties: simple keyword presence
    for kw in PENALTY_KEYWORDS:
        if kw in text:
            out["penalties"].append({
                "penalty_type": kw,
                "raw": kw,
                "article_id": article_id
            })

    # Exceptions: keyword presence (capture the sentence containing the keyword)
    for kw in EXCEPTION_KEYWORDS:
        if kw in text:
            # try to extract the clause containing the keyword
            m = re.search(r'([^\n。؟.!]*' + re.escape(kw) + r'[^\n。؟.!]*)', text)
            clause = m.group(0).strip() if m else kw
            out["exceptions"].append({
                "exception_text": clause,
                "article_id": article_id
            })

    # References: find occurrences like "المادة 17" and produce to_article_id using law_name
    for m in re.findall(REFERENCE_PATTERN, text):
        num_m = re.search(r"\d+", m)
        if num_m:
            to_id = f"{law_name}article{num_m.group()}"
            out["references"].append({
                "from_article_id": article_id,
                "to_article_id": to_id,
                "reference_text": m
            })

    # Concepts: heuristic - look for common legal terms or definitional markers
    for term in LEGAL_TERMS:
        if term in text:
            out["concepts"].append(term)

    # also detect definitional markers "يعني" / "يقصد"
    if re.search(r'\b(يعني|يقصد)\b', text):
        snippet = text[:120].replace('\n', ' ')
        out["concepts"].append(snippet)

    return out


# ------------------------------
def extract_articles(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extract articles from a document text.
    Returns list of article dicts:
      { article_id, article_number, law_name, text, penalties, exceptions, references, legal_concepts }
    """
    text = doc.get("text", "") or ""
    law_name = doc.get("law_name", Path(doc.get("path","")).stem)
    # robust pattern for article headers
    header_pattern = re.compile(r'((?:المواد|المادة|مادة)\s*\d+)', flags=re.IGNORECASE)
    parts = re.split(header_pattern, text)

    articles: List[Dict[str, Any]] = []

    if len(parts) >= 3:
        # parts: [prelude, header1, body1, header2, body2, ...]
        for i in range(1, len(parts), 2):
            header = parts[i].strip()
            body = parts[i+1].strip() if i+1 < len(parts) else ""
            num_match = re.search(r'\d+', header)
            article_num = num_match.group(0) if num_match else str(i // 2)
            article_id = f"{law_name}article{article_num}"
            art = {
                "article_id": article_id,
                "article_number": article_num,
                "law_name": law_name,
                "text": body
            }
            ents = extract_entities_from_text(art)
            art["penalties"] = ents.get("penalties", [])
            art["exceptions"] = ents.get("exceptions", [])
            art["references"] = ents.get("references", [])
            art["legal_concepts"] = ents.get("concepts", [])
            articles.append(art)
    else:
        # fallback: treat whole document as a single article
        article_id = f"{law_name}_full"
        art = {
            "article_id": article_id,
            "article_number": None,
            "law_name": law_name,
            "text": text
        }
        ents = extract_entities_from_text(art)
        art["penalties"] = ents.get("penalties", [])
        art["exceptions"] = ents.get("exceptions", [])
        art["references"] = ents.get("references", [])
        art["legal_concepts"] = ents.get("concepts", [])
        articles.append(art)

    return articles


# ------------------------------
def extract_rules_from_article(article: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Heuristic: split article into candidate rule clauses.
    Returns list of rule dicts:
      { rule_id, rule_text, concepts, penalties, exceptions, references, article_id }
    """
    text = article.get("text", "")
    law_name = article.get("law_name", "")
    rules: List[Dict[str, Any]] = []

    # split by semicolon/newline/Arabic semicolon etc.
    parts = [p.strip() for p in re.split(r'[؛\n]+', text) if p.strip()]
    if len(parts) <= 1:
        # fallback to sentence split
        parts = [s.strip() for s in re.split(SENTENCE_SPLIT, text) if s.strip()]

    # merge very short pieces with previous
    merged_parts = []
    for p in parts:
        if len(p) < 40 and merged_parts:
            merged_parts[-1] = merged_parts[-1] + " " + p
        else:
            merged_parts.append(p)

    for p in merged_parts:
        # create a temporary minimal article dict for entity extraction
        tmp = {"text": p, "article_id": article.get("article_id"), "law_name": law_name}
        ents = extract_entities_from_text(tmp)
        rid = f"{article.get('article_id')}_rule_{uuid.uuid4().hex[:8]}"
        rules.append({
            "rule_id": rid,
            "rule_text": p,
            "concepts": ents.get("concepts", []),
            "penalties": ents.get("penalties", []),
            "exceptions": ents.get("exceptions", []),
            "references": [r.get("to_article_id") for r in ents.get("references", [])],
            "article_id": article.get("article_id")
        })

    if not rules:
        tmp = {"text": text, "article_id": article.get("article_id"), "law_name": law_name}
        ents = extract_entities_from_text(tmp)
        rid = f"{article.get('article_id')}_rule_{uuid.uuid4().hex[:8]}"
        rules.append({
            "rule_id": rid,
            "rule_text": text,
            "concepts": ents.get("concepts", []),
            "penalties": ents.get("penalties", []),
            "exceptions": ents.get("exceptions", []),
            "references": [r.get("to_article_id") for r in ents.get("references", [])],
            "article_id": article.get("article_id")
        })

    return rules


# ------------------------------
def build_index_from_dir(data_dir: str) -> Dict[str, Any]:
    docs = []
    for fname in sorted(os.listdir(data_dir)):
        path = os.path.join(data_dir, fname)
        if not (path.lower().endswith(".pdf") or path.lower().endswith(".txt")):
            continue
        doc = load_document(path)
        if doc:
            docs.append(doc)

    index: Dict[str, Any] = {
        "laws": [], "sections": [], "articles": [],
        "legal_concepts": [], "penalties": [], "exceptions": [], "references": []
    }

    law_seen: Dict[str, str] = {}
    for doc in tqdm(docs, desc="Processing documents"):
        law_name = doc["law_name"]
        law_id = law_name.replace(" ", "_")
        if law_name not in law_seen:
            law_obj = {"law_id": law_id, "law_name": law_name}
            index["laws"].append(law_obj)
            law_seen[law_name] = law_id

        articles = extract_articles(doc)
        for art in articles:
            # ensure article_id normalization
            if not art.get("article_id"):
                art["article_id"] = f"{law_id}article{art.get('article_number') or uuid.uuid4().hex[:6]}"
            # attach law id
            art["law_id"] = law_seen[law_name]

            index["articles"].append({
                "article_id": art["article_id"],
                "article_number": art.get("article_number"),
                "law_id": art.get("law_id"),
                "law_name": art.get("law_name"),
                "article_text": art.get("text"),
                # keep original extracted entities for later use
                "penalties": art.get("penalties", []),
                "exceptions": art.get("exceptions", []),
                "legal_concepts": art.get("legal_concepts", []),
                "references": art.get("references", []),
            })

            # extract rules and add their entities to the index
            rules = extract_rules_from_article(art)
            for r in rules:
                # concepts (strings)
                for c in r.get("concepts", []):
                    concept_id = f"concept_{abs(hash(c))}"
                    index["legal_concepts"].append({
                        "concept_id": concept_id,
                        "concept_name": c,
                        "concept_type": None,
                        "description": None,
                        "article_id": r.get("article_id")
                    })
                # penalties (dicts)
                for p in r.get("penalties", []):
                    penalty_id = f"penalty_{uuid.uuid4().hex[:8]}"
                    index["penalties"].append({
                        "penalty_id": penalty_id,
                        "penalty_type": p.get("penalty_type"),
                        "min_value": p.get("value"),
                        "max_value": p.get("value"),
                        "unit": None,
                        "article_id": r.get("article_id"),
                        "raw": p.get("raw")
                    })
                # exceptions (dicts)
                for ex in r.get("exceptions", []):
                    exception_id = f"exception_{uuid.uuid4().hex[:8]}"
                    index["exceptions"].append({
                        "exception_id": exception_id,
                        "exception_text": ex.get("exception_text") if isinstance(ex, dict) else str(ex),
                        "exception_type": None,
                        "article_id": r.get("article_id")
                    })
                # references (strings of target article ids)
                for ref in r.get("references", []):
                    index["references"].append({
                        "from_article_id": r.get("article_id"),
                        "to_article_id": ref,
                        "reference_text": None
                    })

    # deduplicate helper
    def dedupe(list_of_dicts: List[Dict[str, Any]], key: str) -> List[Dict[str, Any]]:
        seen = set()
        out = []
        for d in list_of_dicts:
            k = d.get(key)
            if k and k not in seen:
                seen.add(k)
                out.append(d)
        return out

    index["legal_concepts"] = dedupe(index["legal_concepts"], "concept_id")
    index["penalties"] = dedupe(index["penalties"], "penalty_id")
    index["exceptions"] = dedupe(index["exceptions"], "exception_id")

    # dedupe references by tuple
    unique_refs = set()
    out_refs = []
    for r in index["references"]:
        tup = (r.get("from_article_id"), r.get("to_article_id"))
        if tup not in unique_refs:
            unique_refs.add(tup)
            out_refs.append(r)
    index["references"] = out_refs

    return index

In [95]:
class Embedder:
    """Generate embeddings"""
    
    def __init__(self):
        print(f"📊 Loading embedding model: {EMBEDDING_MODEL_NAME}")
        self.model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=EMBEDDING_DEVICE)
        print("✅ Model loaded")
    
    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        """Embed multiple texts"""
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        return [emb.tolist() for emb in embeddings]
    
    def embed_single(self, text: str) -> List[float]:
        """Embed single text"""
        return self.model.encode(text, convert_to_numpy=True).tolist()

In [96]:
# ------------------------------
class KnowledgeGraph:
    """Neo4j knowledge graph manager (Legal KG Only)"""

    def __init__(self, uri: str = NEO4J_URI, username: str = NEO4J_USERNAME, password: str = NEO4J_PASSWORD):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))

    def close(self):
        self.driver.close()

    # =========================
    # SCHEMA
    # =========================
    def create_schema(self):
        constraints = [
            "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Law) REQUIRE l.law_id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Section) REQUIRE s.section_id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Article) REQUIRE a.article_id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (c:LegalConcept) REQUIRE c.concept_id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Penalty) REQUIRE p.penalty_id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (e:Exception) REQUIRE e.exception_id IS UNIQUE",
        ]
        with self.driver.session() as session:
            for c in constraints:
                session.run(c)

    # =========================
    # INSERT DATA
    # =========================

    def insert_data(self, index: Dict[str, Any]):
        """Robust insertion for legal data into Neo4j.
        Handles mixed types in index lists (strings or dicts) for concepts/penalties/exceptions/references.
        """
        print("📥 Inserting legal data into Neo4j...")
        with self.driver.session() as session:

            # -------- Laws --------
            law_name_to_id = {}
            for law in index.get("laws", []):
                law_id = law.get("law_id") if isinstance(law, dict) else None
                law_name = law.get("law_name") if isinstance(law, dict) else (str(law) if law else None)
                if not law_id:
                    law_id = (law_name or "UNKNOWN").replace(" ", "_")
                session.run(
                    """
                    MERGE (l:Law {law_id:$law_id})
                    SET l.law_name = coalesce($law_name, l.law_name),
                        l.law_number = coalesce($law_number, l.law_number),
                        l.law_year = coalesce($law_year, l.law_year),
                        l.law_type = coalesce($law_type, l.law_type),
                        l.description = coalesce($description, l.description)
                    """,
                    law_id=law_id,
                    law_name=law.get("law_name") if isinstance(law, dict) else law_name,
                    law_number=law.get("law_number") if isinstance(law, dict) else None,
                    law_year=law.get("law_year") if isinstance(law, dict) else None,
                    law_type=law.get("law_type") if isinstance(law, dict) else None,
                    description=law.get("description") if isinstance(law, dict) else None,
                )
                if law_name:
                    law_name_to_id[law_name] = law_id

            # -------- Sections --------
            for section in index.get("sections", []):
                # section may be dict; skip invalid entries
                if not isinstance(section, dict):
                    continue
                section_id = section.get("section_id") or f"{section.get('law_id','')}_sec_{section.get('section_number','')}"
                session.run(
                    """
                    MERGE (s:Section {section_id:$section_id})
                    SET s.section_name = coalesce($section_name, s.section_name),
                        s.section_number = coalesce($section_number, s.section_number),
                        s.section_type = coalesce($section_type, s.section_type)
                    WITH s
                    MATCH (l:Law {law_id:$law_id})
                    MERGE (l)-[:HAS_SECTION]->(s)
                    """,
                    section_id=section_id,
                    section_name=section.get("section_name"),
                    section_number=section.get("section_number"),
                    section_type=section.get("section_type"),
                    law_id=section.get("law_id") or law_name_to_id.get(section.get("law_name")),
                )

            # -------- Articles --------
            for article in tqdm(index.get("articles", []), desc="Insert Articles"):
                # support either dict entries or preformatted dict-like objects
                if not isinstance(article, dict):
                    continue

                law_id = article.get("law_id")
                law_name = article.get("law_name") or article.get("law")
                if not law_id and law_name:
                    law_id = law_name_to_id.get(law_name)
                if not law_id:
                    law_id = "UNKNOWN_LAW"
                    session.run(
                        "MERGE (l:Law {law_id:$law_id}) SET l.law_name = coalesce(l.law_name, 'UNKNOWN')",
                        law_id=law_id,
                    )

                article_id = article.get("article_id")
                if not article_id:
                    art_num = str(article.get("article_number") or "")
                    article_id = f"{law_id}_article_{art_num or uuid.uuid4().hex[:6]}"

                session.run(
                    """
                    MERGE (a:Article {article_id:$article_id})
                    SET a.article_number = coalesce($article_number, a.article_number),
                        a.article_text = coalesce($article_text, $text, a.article_text),
                        a.article_type = coalesce($article_type, a.article_type)
                    WITH a
                    MATCH (l:Law {law_id:$law_id})
                    MERGE (l)-[:HAS_ARTICLE]->(a)
                    """,
                    article_id=article_id,
                    article_number=article.get("article_number"),
                    article_text=article.get("article_text"),
                    text=article.get("text"),
                    article_type=article.get("article_type"),
                    law_id=law_id,
                )

            # -------- Legal Concepts --------
            for concept in index.get("legal_concepts", []):
                # concept may be a string or a dict
                if isinstance(concept, str):
                    concept_name = concept
                    article_id = None
                    description = None
                    concept_id = f"concept_{abs(hash(concept_name))}"
                elif isinstance(concept, dict):
                    concept_name = concept.get("concept_name") or concept.get("name") or str(concept)
                    article_id = concept.get("article_id")
                    description = concept.get("description")
                    concept_id = concept.get("concept_id") or f"concept_{abs(hash(concept_name))}"
                else:
                    # unknown type, skip
                    continue

                session.run(
                    """
                    MERGE (c:LegalConcept {concept_id:$concept_id})
                    SET c.concept_name = coalesce(c.concept_name, $concept_name),
                        c.description = coalesce(c.description, $description)
                    """,
                    concept_id=concept_id,
                    concept_name=concept_name,
                    description=description,
                )

                if article_id:
                    session.run(
                        """
                        MATCH (a:Article {article_id:$article_id}), (c:LegalConcept {concept_id:$concept_id})
                        MERGE (a)-[:DEFINES]->(c)
                        """,
                        article_id=article_id,
                        concept_id=concept_id,
                    )

            # -------- Penalties --------
            for penalty in index.get("penalties", []):
                if isinstance(penalty, str):
                    penalty_type = penalty
                    article_id = None
                    raw = penalty
                    penalty_id = f"penalty_{abs(hash(penalty_type))}"
                    min_value = None
                    max_value = None
                elif isinstance(penalty, dict):
                    penalty_type = penalty.get("penalty_type") or str(penalty)
                    article_id = penalty.get("article_id")
                    raw = penalty.get("raw") or penalty_type
                    penalty_id = penalty.get("penalty_id") or f"penalty_{uuid.uuid4().hex[:8]}"
                    min_value = penalty.get("min_value")
                    max_value = penalty.get("max_value")
                else:
                    continue

                session.run(
                    """
                    MERGE (p:Penalty {penalty_id:$penalty_id})
                    SET p.penalty_type = coalesce(p.penalty_type, $penalty_type),
                        p.min_value = coalesce($min_value, p.min_value),
                        p.max_value = coalesce($max_value, p.max_value),
                        p.unit = coalesce($unit, p.unit),
                        p.raw = coalesce($raw, p.raw)
                    """,
                    penalty_id=penalty_id,
                    penalty_type=penalty_type,
                    min_value=min_value,
                    max_value=max_value,
                    unit=penalty.get("unit") if isinstance(penalty, dict) else None,
                    raw=raw,
                )

                if isinstance(penalty, dict) and penalty.get("article_id"):
                    session.run(
                        """
                        MATCH (a:Article {article_id:$article_id}), (p:Penalty {penalty_id:$penalty_id})
                        MERGE (a)-[:HAS_PENALTY]->(p)
                        """,
                        article_id=penalty.get("article_id"),
                        penalty_id=penalty_id,
                    )

            # -------- Exceptions --------
            for exception in index.get("exceptions", []):
                if isinstance(exception, str):
                    exception_text = exception
                    article_id = None
                    exception_id = f"exception_{abs(hash(exception_text))}"
                elif isinstance(exception, dict):
                    exception_text = exception.get("exception_text") or str(exception)
                    article_id = exception.get("article_id")
                    exception_id = exception.get("exception_id") or f"exception_{uuid.uuid4().hex[:8]}"
                else:
                    continue

                session.run(
                    """
                    MERGE (e:Exception {exception_id:$exception_id})
                    SET e.exception_text = coalesce(e.exception_text, $exception_text),
                        e.exception_type = coalesce($exception_type, e.exception_type)
                    """,
                    exception_id=exception_id,
                    exception_text=exception_text,
                    exception_type=exception.get("exception_type") if isinstance(exception, dict) else None,
                )

                if isinstance(exception, dict) and exception.get("article_id"):
                    session.run(
                        """
                        MATCH (a:Article {article_id:$article_id}), (e:Exception {exception_id:$exception_id})
                        MERGE (a)-[:HAS_EXCEPTION]->(e)
                        """,
                        article_id=exception.get("article_id"),
                        exception_id=exception_id,
                    )

            # -------- References --------
            for ref in index.get("references", []):
                # expect dicts with from_article_id / to_article_id, otherwise skip
                if not isinstance(ref, dict):
                    continue
                from_id = ref.get("from_article_id")
                to_id = ref.get("to_article_id")
                if not from_id or not to_id:
                    continue
                session.run(
                    """
                    MATCH (a1:Article {article_id:$from_id})
                    MATCH (a2:Article {article_id:$to_id})
                    MERGE (a1)-[r:REFERS_TO]->(a2)
                    SET r.text = coalesce(r.text, $ref_text)
                    """,
                    from_id=from_id,
                    to_id=to_id,
                    ref_text=ref.get("reference_text"),
                )

        print("✅ Legal data inserted successfully")

    # =========================
    # CLEAR GRAPH
    # =========================
    def clear_all(self):
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")

In [101]:
# ------------------------------
def load_all_documents(data_folder: str) -> List[Dict[str, Any]]:
    print(f"\n📂 Loading documents from: {data_folder}")
    
    documents = []
    data_path = Path(data_folder)

    if not data_path.exists():
        raise FileNotFoundError(f"Data folder not found: {data_folder}")

    for file in data_path.glob("**/*"):
        if file.suffix.lower() in [".pdf", ".txt"]:
            doc = load_document(str(file))
            if doc:
                documents.append(doc)
                print(f"  ✓ {file.name}")

    if not documents:
        raise ValueError("No documents found")

    print(f"\n✅ Loaded {len(documents)} documents")
    return documents


def extract_full_legal_index(documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    print("\n📄 Extracting legal structure...")

    all_articles = []
    all_laws = {}

    all_concepts = []
    all_penalties = []
    all_exceptions = []
    all_references = []

    for doc in documents:
        articles = extract_articles(doc)

        for art in articles:
            all_articles.append(art)

            law_name = art.get("law_name")
            if law_name and law_name not in all_laws:
                all_laws[law_name] = {
                    "law_id": law_name.replace(" ", "_"),
                    "law_name": law_name,
                }

            # collect entities if present
            all_concepts.extend(art.get("legal_concepts", []))
            all_penalties.extend(art.get("penalties", []))
            all_exceptions.extend(art.get("exceptions", []))
            all_references.extend(art.get("references", []))

    index = {
        "laws": list(all_laws.values()),
        "sections": [],
        "articles": all_articles,
        "legal_concepts": all_concepts,
        "penalties": all_penalties,
        "exceptions": all_exceptions,
        "references": all_references,
    }

    print(f"✅ Laws: {len(index['laws'])}")
    print(f"✅ Articles: {len(index['articles'])}")
    print(f"✅ Concepts: {len(index['legal_concepts'])}")
    print(f"✅ Penalties: {len(index['penalties'])}")
    print(f"✅ Exceptions: {len(index['exceptions'])}")
    print(f"✅ References: {len(index['references'])}")

    return index


def build_neo4j_legal_graph(index: Dict[str, Any]) -> Dict[str, int]:
    print("\n🕸️  Building Neo4j Legal Knowledge Graph...")

    graph = KnowledgeGraph()

    graph.create_schema()
    graph.clear_all()
    graph.insert_data(index)

    stats = None
    graph.close()

    return stats

In [ ]:
def run_build_pipeline(data_folder: str):
    print("=" * 80)
    print("🔥 BUILDING ARABIC LEGAL KNOWLEDGE GRAPH")
    print("=" * 80)

    documents = load_all_documents(data_folder)

    index = extract_full_legal_index(documents)

    stats = build_neo4j_legal_graph(index)

    print("\n" + "=" * 80)
    print("✅ BUILD COMPLETE!")
    print("=" * 80)

In [103]:
run_build_pipeline("Datasets/Unstructured_Data")

🔥 BUILDING ARABIC LEGAL KNOWLEDGE GRAPH

📂 Loading documents from: Datasets/Unstructured_Data
  ✓ 3okobat.pdf
  ✓ 3okobat_num36_2020.txt
  ✓ 3okobat_num6_2020.txt
  ✓ asle7a.txt
  ✓ child.txt
  ✓ dostor.pdf
  ✓ drugs.txt
  ✓ egra2at_gena2ya.pdf
  ✓ erhab.txt
  ✓ mbade2_ma7kmt_elna2d.pdf
  ✓ tech.txt

✅ Loaded 11 documents

📄 Extracting legal structure...
✅ Laws: 11
✅ Articles: 1081
✅ Concepts: 251
✅ Penalties: 1735
✅ Exceptions: 576
✅ References: 29

🕸️  Building Neo4j Legal Knowledge Graph...
📥 Inserting legal data into Neo4j...


Insert Articles: 100%|██████████| 1081/1081 [02:30<00:00,  7.17it/s]


✅ Legal data inserted successfully

✅ BUILD COMPLETE!
📊 Neo4j Statistics:


AttributeError: 'NoneType' object has no attribute 'get'